In [2]:
import torch
print(torch.cuda.is_available())

True


In [3]:
!pip install -q "sentence-transformers>=3.0" datasets transformers accelerate


In [1]:
# ============================================================
# Cell 1 — Environment setup
# Run this cell FIRST after a fresh kernel restart.
# A kernel restart is mandatory if you previously hit OOM —
# PyTorch's CUDA caching allocator cannot release memory back
# to the OS without a full process reset.
# ============================================================
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Must be set BEFORE torch is imported for it to take effect.

import gc
import json
import torch

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {free/1e9:.2f} GB free / {total/1e9:.2f} GB total")

# Abort early if VRAM isn't clean — better to know now than mid-training.
assert free / total > 0.85, (
    "Less than 85 % VRAM free — restart the Colab runtime "
    "(Runtime → Restart session) and re-run from this cell."
)


# ============================================================
# Cell 2 — Imports
# ============================================================
from datasets import Dataset
from sentence_transformers import CrossEncoder
from sentence_transformers.cross_encoder import (
    CrossEncoderTrainer,
    CrossEncoderTrainingArguments,
)
from sentence_transformers.cross_encoder.losses import MultipleNegativesRankingLoss


# ============================================================
# Cell 3 — Load hard-negative pairs
# ============================================================
with open("mnrl_finetuning_pairs.json", "r") as f:
    raw = json.load(f)

dataset_dict = {
    "query":      raw["queries"],
    "positive":   raw["positives"],
    "negative_1": raw["hard_negatives_1"],
    "negative_2": raw["hard_negatives_2"],
}

hf_dataset = Dataset.from_dict(dataset_dict)
print(hf_dataset)


# ============================================================
# Cell 4 — Hyperparameters (tuned for Tesla T4, 15 GB)
# ============================================================
MODEL_NAME  = "allenai/scibert_scivocab_uncased"
OUTPUT_DIR  = "scibert-cross-encoder-reranker"

# ── sequence length ────────────────────────────────────────
# BERT attention memory ∝ seq_len². Halving 512→256 cuts the
# attention tensor to 1/4 the size. Biomedical title+abstract
# pairs typically fit their key signal within 256 tokens.
MAX_SEQ_LEN = 256

# ── batch / accumulation ──────────────────────────────────
# MNRL fires (1 + num_negatives) = 3 forward passes per step.
# T4 budget per pass at batch=4, seq=256, FP16 ≈ 1.5 GB
# → 3 passes ≈ 4.5 GB active + 4.5 GB gradients + 1.7 GB model
# → safe within 15 GB with headroom.
#
# Gradient accumulation recovers the effective batch size:
#   effective_batch = BATCH_SIZE × GRAD_ACCUM = 4 × 8 = 32
#   in-batch negatives per query = (32 - 1) + 2 hard = 33
BATCH_SIZE  = 4
GRAD_ACCUM  = 8

NUM_EPOCHS    = 3
LEARNING_RATE = 2e-5


# ============================================================
# Cell 5 — Model  (replace the previous Cell 5)
# ============================================================
# WHY NO model_kwargs torch_dtype here:
# Loading in FP16 conflicts with AMP (fp16=True in training args).
# AMP's grad scaler requires FP32 gradients to unscale — if the
# model params are FP16, the gradients are also FP16, which raises
# "Attempting to unscale FP16 gradients."
#
# The correct pattern is:
#   - Load model in FP32 (default)              → ~1.7 GB static
#   - Enable fp16=True in TrainingArguments      → AMP casts
#     activations to FP16 during forward pass,
#     keeps optimizer states in FP32 (standard
#     mixed-precision training)
#
# Net result: same FP16 arithmetic speed benefit, no conflict.
# ============================================================

model = CrossEncoder(
    MODEL_NAME,
    max_length=MAX_SEQ_LEN,   # 256, defined in Cell 4
    num_labels=1,
    # No model_kwargs / torch_dtype — let it default to FP32
)

model.model.gradient_checkpointing_enable()

free, total = torch.cuda.mem_get_info()
print(f"After model load — VRAM free: {free/1e9:.2f} GB / {total/1e9:.2f} GB")
# Expect ~13.9 GB free (model ~1.7 GB in FP32)
# AMP will keep activations in FP16 during training,
# so peak VRAM during a forward pass stays well under 15 GB.

# ============================================================
# Cell 6 — Loss
# ============================================================
loss = MultipleNegativesRankingLoss(model, num_negatives=2)


# ============================================================
# Cell 7 — Training arguments
# ============================================================
training_args = CrossEncoderTrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LEARNING_RATE,
    warmup_steps=0.1,       # float → interpreted as ratio in ST v3+
    weight_decay=0.01,

    fp16=True,              # AMP keeps forward activations in FP16;
                            # optimizer states stay in FP32 (standard AMP)

    # Pinned memory pre-loads batches into page-locked RAM; on
    # Colab this competes directly with GPU VRAM — disable it.
    dataloader_pin_memory=False,

    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=2,
    seed=42,
)


# ============================================================
# Cell 8 — Train
# ============================================================
trainer = CrossEncoderTrainer(
    model=model,
    args=training_args,
    train_dataset=hf_dataset,
    loss=loss,
)

print("=== Training config ===")
print(f"  max_length           : {MAX_SEQ_LEN}")
print(f"  batch_size           : {BATCH_SIZE}")
print(f"  grad_accum_steps     : {GRAD_ACCUM}")
print(f"  effective_batch      : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  in-batch negatives   : {BATCH_SIZE * GRAD_ACCUM - 1} + 2 hard = "
      f"{BATCH_SIZE * GRAD_ACCUM + 1} per query")
print(f"  gradient_checkpnt    : enabled")
print(f"  model dtype at load  : float16")
print(f"  fp16 AMP             : enabled")
print()

trainer.train()
print("Fine-tuning complete.")


# ============================================================
# Cell 9 — Save
# ============================================================
model.save(OUTPUT_DIR)
print(f"Saved to: {OUTPUT_DIR}/")


# ============================================================
# Cell 10 — Sanity check
# ============================================================
test_pairs = [
    ["What is the role of p53 in apoptosis?",
     "Title: TP53 mutations in cancer Abstract: p53 is a tumour suppressor "
     "that regulates apoptosis through transcriptional activation of pro-apoptotic genes."],
    ["What is the role of p53 in apoptosis?",
     "Title: Climate variability Abstract: El Niño events drive shifts in "
     "global precipitation patterns and surface temperatures."],
]

scores = model.predict(test_pairs)
print("\nSanity-check scores (higher = more relevant):")
for pair, score in zip(test_pairs, scores):
    print(f"  {score:+.4f}  |  '{pair[1][:70]}…'")

assert scores[0] > scores[1], "Reranker should score the relevant doc higher!"
print("\nSanity check passed ✓")

PyTorch  : 2.11.0+cu128
CUDA     : True
GPU      : Tesla T4
VRAM     : 15.53 GB free / 15.64 GB total
Dataset({
    features: ['query', 'positive', 'negative_1', 'negative_2'],
    num_rows: 1964
})


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning complete.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: scibert-cross-encoder-reranker/

Sanity-check scores (higher = more relevant):
  +0.8114  |  'Title: TP53 mutations in cancer Abstract: p53 is a tumour suppressor t…'
  +0.0654  |  'Title: Climate variability Abstract: El Niño events drive shifts in gl…'

Sanity check passed ✓
